# 02. Gaussian Dual Adjacency Graph (GDAG) Construction
**Project:** *Does Bangalore Traffic Explain House Prices? Uncertainty-Aware Price Prediction with Road-Network GNNs and Bayesian Heads*  
**Paper Grounding:** Ravichandran et al. (*Nature Scientific Reports*, 2025, 15:41208)  
**Method:** Algorithm 2: Gaussian Dual Adjacency Graph (GDAG) & Dynamic Adjacency Matrix

---
### Objectives:
1. Construct the **Static Adjacency Matrix** $A_{	ext{static}}$ from physical road connectivity.
2. Compute the **Gaussian RBF Dynamic Adjacency Matrix** $A_{	ext{dynamic}}$ based on traffic congestion profiles.
3. Formulate the **Dual Adjacency Matrix** $DAM = A_{	ext{static}} + A_{	ext{dynamic}}$ and its normalized Laplacian.
4. Visualize how $DAM$ creates semantic traffic bridges between disjoint bottleneck corridors in Bangalore (e.g. Silk Board and Marathahalli).

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

from src.logger import ServerLogger
from src.gdag_spatial_graph import GaussianDualAdjacencyGraph

logger = ServerLogger.get_logger()
logger.info("Notebook 02: GDAG Spatial Graph initialized.")

## 1. Load Locality Nodes and Road Graph Edges

In [ ]:
df_nodes = pd.read_csv('../data/bangalore_locality_graph_nodes.csv')
df_edges = pd.read_csv('../data/bangalore_locality_graph_edges.csv')

print(f"Number of canonical locality nodes: {len(df_nodes)}")
print(f"Number of physical road edges:       {len(df_edges)}")
df_nodes.head(4)

## 2. Compute Gaussian Dual Adjacency Matrix (DAM)
$$A_{	ext{dynamic}}(i, j) = \exp\left(-rac{\|x_i - x_j\|^2}{2\sigma^2}ight)$$
$$DAM = A_{	ext{static}} + A_{	ext{dynamic}}$$
$$\widetilde{DAM} = D^{-1/2}(DAM) \cdot DAM \cdot D^{-1/2}(DAM)$$

In [ ]:
gdag = GaussianDualAdjacencyGraph(sigma=1.2, dynamic_threshold=0.30)
DAM, norm_DAM = gdag.build_dual_adjacency_matrix(df_nodes, df_edges)

print(f"Dual Adjacency Matrix shape: {DAM.shape}")
print(f"Non-zero elements:            {np.count_nonzero(DAM > 0.05)}")
print(f"Matrix density:               {np.count_nonzero(DAM > 0.05) / (DAM.shape[0]**2)*100:.2f}%")

## 3. Adjacency Heatmap Visualization: Static vs Dynamic vs Dual

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Sample 30 localities for clean visualization
sample_n = 30
sns.heatmap(norm_DAM[:sample_n, :sample_n], ax=axes[0], cmap='Blues', cbar=False)
axes[0].set_title(f"Normalized DAM (First {sample_n} Localities)", fontsize=11)

# Degree distribution of DAM
deg = np.sum(DAM > 0.1, axis=1)
sns.histplot(deg, kde=True, ax=axes[1], color='teal')
axes[1].set_title("Locality Node Degree Distribution in GDAG", fontsize=11)
axes[1].set_xlabel("Degree (Connected Localities)")

# Eigenvalue Spectrum of Normalized DAM
eigenvalues = np.linalg.eigvalsh(norm_DAM)
axes[2].plot(np.sort(eigenvalues), marker='o', color='purple', markersize=3)
axes[2].set_title("Graph Laplacian Eigenvalue Spectrum", fontsize=11)
axes[2].set_xlabel("Eigenvalue Index")
axes[2].set_ylabel("Eigenvalue Magnitude")

plt.tight_layout()
plt.show()

## 4. Convert GDAG to PyTorch Geometric Tensors

In [ ]:
feature_cols = [
    'latitude', 'longitude', 'node_degree',
    'commute_whitefield_peak_min', 'commute_bellandur_peak_min',
    'commute_ecity_peak_min', 'commute_manyata_peak_min',
    'commute_mgroad_peak_min', 'avg_congestion_ratio',
    'dist_to_metro_km', 'commute_metro_access_min'
]
x_tensor, edge_index, edge_attr = gdag.to_pyg_tensors(df_nodes, feature_cols)

print(f"Node feature tensor shape: {x_tensor.shape}")
print(f"Edge index tensor shape:    {edge_index.shape}")
print(f"Edge attr tensor shape:     {edge_attr.shape}")
logger.info("Notebook 02 execution completed. PyG tensors validated.")